# Y-ZA YOLOv8n on ExDark (Google Colab)

Trains the baseline YOLOv8n and Y-ZA YOLOv8n (ECA + GC in the neck) with identical settings and prints both mAP@0.5 scores.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Put `YZA_YOLOv8n.zip` (the project zip) in your Google Drive, in *My Drive*.
3. Download the ExDark dataset from the official page (github.com/cs-chan/Exclusively-Dark-Image-Dataset) and put the folder in *My Drive* so it contains `ExDark/` and `ExDark_Anno/` (and `imageclasslist.txt` if provided).

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- EDIT THESE TWO PATHS IF YOURS DIFFER ---
PROJECT_ZIP = '/content/drive/MyDrive/YZA_YOLOv8n.zip'
EXDARK_ROOT = '/content/drive/MyDrive/ExDark_root'   # folder that contains ExDark/ and ExDark_Anno/

!rm -rf /content/yza && mkdir -p /content/yza
!unzip -q -o "{PROJECT_ZIP}" -d /content/yza
%cd /content/yza
!python convert_exdark.py --src "{EXDARK_ROOT}" --dst /content/yza/datasets/ExDark

In [ ]:
# write a data.yaml with absolute paths for this runtime
names = ["Bicycle","Boat","Bottle","Bus","Car","Cat","Chair","Cup","Dog","Motorbike","People","Table"]
with open('data.yaml', 'w') as f:
    f.write('path: /content/yza/datasets/ExDark\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n')
    for i, n in enumerate(names):
        f.write(f'  {i}: {n}\n')
print(open('data.yaml').read())

In [ ]:
!python check_model.py

## Train the baseline (50 epochs, batch 8)

In [ ]:
!python train_baseline.py

## Train Y-ZA YOLOv8n (50 epochs, batch 8)

In [ ]:
!python train_yza.py

## Compare results

In [ ]:
import glob, pandas as pd
for path in sorted(glob.glob('runs/**/results.csv', recursive=True)):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    best = df['metrics/mAP50(B)'].max()
    print(f'{path}: best mAP@0.5 = {best:.3f} (epochs run: {len(df)})')

In [ ]:
# save everything back to Google Drive so you do not lose it
!mkdir -p /content/drive/MyDrive/YZA_results
!cp -r runs /content/drive/MyDrive/YZA_results/